In [ ]:
!pip install dotenv openai requests
#!pip uninstall transformers -y
#!pip install transformers

# Preparing for API connection

You are an expert documentary linguist specializing in morphological inflection in the Kashubian language. You are given the inflected word "<word>". If it is neither a verb, adverb, nor noun, return <invalid>. Otherwise, return its lemma and morphosyntactic features in the UniMorph 4.0 schema, as follows: <lemma> <features>.

In [ ]:
# get api api key, source: https://www.geeksforgeeks.org/python/how-to-create-and-use-env-files-in-python/

# importing os module for environment variables
import os
# importing necessary functions from dotenv library
from dotenv import load_dotenv, dotenv_values 
# loading variables from .env file
load_dotenv(dotenv_path="/dss/dsshome1/03/ge87wod2/.env",override=True) 

# accessing and printing value
#print(os.getenv("GEMINI_API_KEY"))
OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY")

import requests
import json
response = requests.get(
	url="https://openrouter.ai/api/v1/key",
	headers={
		"Authorization": f"Bearer {OPENROUTER_API_KEY}"
	}
)
print(json.dumps(response.json(), indent=2))


{
  "data": {
    "label": "sk-or-v1-25a...d92",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 18,
    "limit_reset": null,
    "limit_remaining": 14.102342364,
    "include_byok_in_limit": false,
    "usage": 3.897657636,
    "usage_daily": 2.146143,
    "usage_weekly": 2.248204652,
    "usage_monthly": 2.619995652,
    "byok_usage": 0.062251124,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "creator_user_id": "user_340MMJCC58N5dKZhT0S2aCVkycf",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}


In [ ]:
# credits before one language
{
	"data": {
		"label": "sk-or-v1-25a...d92",
		"is_provisioning_key": false,
		"limit": 18,
		"limit_reset": null,
		"limit_remaining": 17.838582579,
		"include_byok_in_limit": false,
		"usage": 0.161417421,
		"usage_daily": 0.0385767,
		"usage_weekly": 0.0385767,
		"usage_monthly": 0.161417421,
		"byok_usage": 0,
		"byok_usage_daily": 0,
		"byok_usage_weekly": 0,
		"byok_usage_monthly": 0,
		"is_free_tier": false,
		"expires_at": null,
		"rate_limit": {
			"requests": -1,
			"interval": "10s",
			"note": "This field is deprecated and safe to ignore."
		}
	}
}
# credits after one language
{
	"data": {
		"label": "sk-or-v1-25a...d92",
		"is_provisioning_key": false,
		"limit": 18,
		"limit_reset": null,
		"limit_remaining": 17.804664679,
		"include_byok_in_limit": false,
		"usage": 0.195335321,
		"usage_daily": 0.0724946,
		"usage_weekly": 0.0724946,
		"usage_monthly": 0.195335321,
		"byok_usage": 0,
		"byok_usage_daily": 0,
		"byok_usage_weekly": 0,
		"byok_usage_monthly": 0,
		"is_free_tier": false,
		"expires_at": null,
		"rate_limit": {
			"requests": -1,
			"interval": "10s",
			"note": "This field is deprecated and safe to ignore."
		}
	}
}

# Zero-Shot Prompting

In [ ]:
model = "google/gemini-2.5-flash"
model = "openai/gpt-oss-120b"
model_name = model.split("/")[-1]
sigmorphon_path = "/dss/dsshome1/03/ge87wod2/morphological-inflection/2023InflectionST/part1/data/"
dataset = ".tst"
data_path = "./data/" + model_name + "/"
lang_code = "deu"
lang = "German"

from openai import OpenAI
client = OpenAI(
		base_url="https://openrouter.ai/api/v1",
		api_key=OPENROUTER_API_KEY,
)
in_path = sigmorphon_path + lang_code + dataset
out_path = data_path + lang_code + "_" + model_name + ".out"
import os
if not os.path.isdir(data_path): os.mkdir(data_path)
with open (in_path,"r") as in_file, open(out_path,"w") as out_file:
				correct = 0
				lines = in_file.readlines()
				line_count = len(lines)
				for line in lines:
					if line == "": continue
					lemma, features, target = line.split("\t")
					while (True):
						response = client.chat.completions.create(
								extra_headers={
								},
								model=model,
								messages=[
										{"role": "system", "content": f"""You are an expert documentary linguist specializing in morphological inflection in the language {lang}. You are given a lemma in {lang} and features in the hierarchical annotation schema UniMorph 4.0. Inflect the lemma according to the features and output only the inflected form."""},
										{"role": "user", "content": f"Inflect lemma + features: {lemma} + {features}"}
								],
						)
						try: 
							pred = (response.choices[0].message).content.strip()
							target = target.strip()
							if pred == target: 
										correct = correct + 1
										print (pred,"is correct.")
							else: print(pred,target)
							out_file.write(lemma + "\t" + features + "\t" + pred + "\n")
							out_file.flush()
							break
						except:	
							print("retry")
				acc = correct / line_count
				out_file.write("test acc "+lang+" " + str(acc)+"\n")	
					
print(acc,lang)

Aikido is correct.
Aikido is correct.
Aikidos is correct.
Aikido is correct.
Anreisen is correct.
Anreise is correct.
Anreisen is correct.
Anreise is correct.


# Generate 11k Kashubian training samples

In [ ]:
model = "openai/gpt-oss-120b"
model = "google/gemini-2.5-flash"
model_name = model.split("/")[-1]
sigmorphon_path = "/dss/dsshome1/03/ge87wod2/morphological-inflection/2023InflectionST/part1/data/"
!tar -xzvf /dss/dsshome1/03/ge87wod2/morphological-inflection/data/leipzig-corpera-collection/csb_wikipedia_2021_10K.tar.gz 
csb_path = "csb_wikipedia_2021_10K/csb_wikipedia_2021_10K-sentences.txt"
dataset = ".tst"
data_path = "./data/" + model_name + "/"
lang_code = "csb"
lang = "Kashubian"
import random
random.seed(10)
import itertools
from openai import OpenAI
client = OpenAI(
		base_url="https://openrouter.ai/api/v1",
		api_key=OPENROUTER_API_KEY,
)

def get_lemma(line):
	lemma = line.split("\t")[0]
	return lemma
def get_pos(line):
	return line.split("\t")[1].split(";")[0]
def get_features(line):
	# print(line)
	features = line.split("\t")[1]
	return features
def get_word(line):
	 word = line.strip().split("\t")[2]
	 return word
import re
def spec_chars(string):
	spec_chars = re.compile('^[@_!#$%^&*()=<>?/\|}{~:]')
	return not (spec_chars.search(string) == None)
def valid(line):
	if (len (line.split("\t"))) == 3:
			lemma = get_lemma(line)
			if lemma == "N": return False
			if len(get_lemma(line).split(" ")) > 1: return False # more than one word lemma
			features = get_features(line)
			if(len(features.split(";"))>4): return False
			word = get_word(line)
			pos = get_pos(line)
			if pos not in ("ADV","V","N","ADJ"): return False
			if spec_chars(lemma) or spec_chars(features) or spec_chars(word): return False
			if ";" in lemma or ";" in word: return False
			if not lemma: return False
			return True 
	return False



in_path = csb_path
in_path_sigmorphon = sigmorphon_path + lang_code + ".tst"
out_path = f"data/{model.split('/')[1]}/csb"
counter = 0
seen_words = []
with open (in_path,"r") as in_file_sent, open(in_path_sigmorphon,"r") as in_file_sigmorphon, open(out_path,"w") as out_file:
	lemmas_sigmorphon = [(k) for k, _ in itertools.groupby(in_file_sigmorphon.readlines(), get_lemma) if k != ""]
	lines = in_file_sent.readlines()
	output = []
	for i in range(len(lines)):
		line = lines[i]
		if line == "": continue
		# print(line.split("\t")[1])
		words = line.split("\t")[1].split(" ") # get words
		words_without_symbols =  ["".join(char for char in word if char.isalnum()) for word in words] # remove special chars
		long_words = list (filter(lambda x: len(x)>3,words_without_symbols)) # get longer words
		# print(long_words)
		random.shuffle(long_words)
		successful = 0
		while ( len(long_words) and successful < 2): # max. two words per sentence
			word = long_words.pop()
			if word in seen_words: continue
			else: seen_words.append(word) # duplicate
			response = client.chat.completions.create(
					model=model,
					messages=[
							{"role": "user", "content": f"""You are an expert documentary linguist specializing in morphological inflection in the Kashubian language. You are given the (inflected) word "{word}". If it is neither a verb, a noun, an adverb, nor an adjective, return <invalid>. Otherwise, return its lemma and morphosyntactic features in the hierarchical feature schema of UniMorph, as follows: <lemma>\t<features>\t<word>. Here is an example of the morphosyntactic features: N;ACC;SG"""}, # one-shot due to model outputting inconsistent feature schema
					],
			)
			try: 
				pred = (response.choices[0].message).content.strip()
				target = pred.strip()
				# print("target",target)
				if "invalid" not in target: 
						length = len(target.split("\t"))
						if (length != 3): continue
						lemma, features, pred = target.split("\t")
						if (lemma in lemmas_sigmorphon or not lemma): 
							print("lemma in test data"); 
							continue
						text = lemma + "\t" + features + "\t" + pred + "\n"
						if not valid(text): 
							print("something wrong with the format"); 
							continue
						out_file.write(text)
						out_file.flush()
						successful += 1
						counter += 1
						print("text",text)
						print("line",i,"counter",counter)
			except Exception as e:	
				print(e)
!rm -rf csb_wikipedia_2021_10K/
# TODO filter out outputs containing /, *,  |, POS wrong, if lemma contains ; and the features do not, then swap lemma and features, Lemma is N or Adjective, word was _, word was ADV, lemma was PRON, maybe account for repeating word forms

<>:34: SyntaxWarning: invalid escape sequence '\|'
<>:34: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_3531828/1005804026.py:34: SyntaxWarning: invalid escape sequence '\|'
  spec_chars = re.compile('^[@_!#$%^&*()=<>?/\|}{~:]')


csb_wikipedia_2021_10K/
csb_wikipedia_2021_10K/csb_wikipedia_2021_10K-inv_w.txt
csb_wikipedia_2021_10K/csb_wikipedia_2021_10K-co_s.txt
csb_wikipedia_2021_10K/csb_wikipedia_2021_10K-sentences.txt
csb_wikipedia_2021_10K/csb_wikipedia_2021_10K-words.txt
csb_wikipedia_2021_10K/csb_wikipedia_2021_10K-import.sql
csb_wikipedia_2021_10K/csb_wikipedia_2021_10K-sources.txt
csb_wikipedia_2021_10K/csb_wikipedia_2021_10K-inv_so.txt
csb_wikipedia_2021_10K/csb_wikipedia_2021_10K-co_n.txt
something wrong with the format
something wrong with the format
text blós	N;MASC;ACC;SG	blós

line 1 counter 1


/tmp/ipykernel_3531828/1005804026.py:34: SyntaxWarning: invalid escape sequence '\|'
  spec_chars = re.compile('^[@_!#$%^&*()=<>?/\|}{~:]')


KeyboardInterrupt: 

In [ ]:
def get_lemma(line):
	lemma = line.split("\t")[0]
	return lemma
def get_features(line):
	print(line)
	features = line.split("\t")[1]
	return features
def get_word(line):
	word = line.strip().split("\t")[2]
	return word
def get_pos(line):
	return line.split("\t")[1].split(";")[0]
import re
def spec_chars(string):
	spec_chars = re.compile('^[@_!#$%^&*()=<>?/\|}{~:]')
	return not (spec_chars.search(string) == None)
def valid(line):
	if (len (line.strip().split("\t"))) == 3:
			lemma = get_lemma(line)
			if lemma == "N": return False
			if len(get_lemma(line).split(" ")) > 1: return False # more than one word lemma
			features = get_features(line)
			word = get_word(line)
			pos = get_pos(line)
			if pos not in ("ADV","V","N","ADJ"): return False
			if spec_chars(lemma) or spec_chars(features) or spec_chars(word): return False
			if ";" in lemma or ";" in word: return False
			if not lemma: 
				return False
			return True 
	return False


def cleaner_features(line):
	features = get_features(line)
	lemma = get_lemma(line).split("/")[0].split("\\")[0]
	word = get_word(line)
	split_features = features.split(";")
	split_features_without_symbols = [f.split("\\")[0].split("/")[0] for f in split_features]
	features_joined_all = []
	for f in split_features_without_symbols:
		feature_parts = re.split(r"[.:()-]", f)	
		feature_parts_upper = [f.upper() for f in feature_parts]
		features_joined = ";".join(feature_parts_upper)
		features_joined_all.append(features_joined)
	merged_features = ";".join(features_joined_all)
	return lemma + "\t" + merged_features + "\t" + word + "\n"

model = "google/gemini-2.5-flash"
in_path = f"data/{model.split('/')[1]}/csb-filter"
out_path = f"data/{model.split('/')[1]}/csb-filter-2"
with open(in_path,"r",encoding="utf-8") as in_file, open(out_path,"w") as out_file:
	lines = in_file.readlines()
	lines_valid = [cleaner_features(l) for l in lines if valid(l)]
	lines_string = "".join(lines_valid)
	out_file.write(lines_string)

<>:16: SyntaxWarning: invalid escape sequence '\|'
<>:16: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_798378/1239856299.py:16: SyntaxWarning: invalid escape sequence '\|'
  spec_chars = re.compile('^[@_!#$%^&*()=<>?/\|}{~:]')


rozestać	V;PST;MASC;SG	rozestôł

rozestać	V;PST;MASC;SG	rozestôł

pózny	ADJ;MASC;VOC;SG	pózni

pózny	ADJ;MASC;VOC;SG	pózni

zwãk	N;MASC;SG;NOM	zwãk

zwãk	N;MASC;SG;NOM	zwãk

ùczbòwnik	N;NOM;PL	ùczbòwniczi

ùczbòwnik	N;NOM;PL	ùczbòwniczi

Ablacëjô	N;NOM;SG;FEM	Ablacëjô

Ablacëjô	N;NOM;SG;FEM	Ablacëjô

cepły	ADJ;FEM;SG;NOM	cepła

cepły	ADJ;FEM;SG;NOM	cepła

panëszka	N;INS;SG;F	panëszką

panëszka	N;INS;SG;F	panëszką

òpisënk	N;NOM;SG	òpisënk

òpisënk	N;NOM;SG	òpisënk

nôbarżi	ADV;SUP	nôbarżi

nôbarżi	ADV;SUP	nôbarżi

jinstrument	N;GEN;SG	jinstrumentu

jinstrument	N;GEN;SG	jinstrumentu

Aborigen	N;NOM;PL	Aborigeni

Aborigen	N;NOM;PL	Aborigeni

rëba	N;NOM;SG;F	rëba

rëba	N;NOM;SG;F	rëba

Abram	N;NOM;SG	Abramis

Abram	N;NOM;SG	Abramis

dzéjnik	N;GEN;PL	dzejników

dzéjnik	N;GEN;PL	dzejników

Realizacja	N;GEN;SG	Realizacji

Realizacja	N;GEN;SG	Realizacji

Telewizjô	N;GEN;SG	Telewizji

Telewizjô	N;GEN;SG	Telewizji

chalëpnik	N;NOM;SG	chalëpnik

chalëpnik	N;NOM;SG	chalëpnik

rëba	N;FEM;GEN;SG	rë

In [2]:
import itertools
import random
random.seed(10)
def get_lemma(line):
	lemma = line.split("\t")[0]
	return lemma
def get_word(line):
	if len(line.strip().split("\t")) != 3:
		return ""
	word = line.strip().split("\t")[2]
	return word

input_path = "/dss/dsshome1/03/ge87wod2/morphological-inflection/llm/data/gemini-2.5-flash/csb-filter-3"
train_path = "/dss/dsshome1/03/ge87wod2/morphological-inflection/llm/data/gemini-2.5-flash/csb.trn"
valid_path = "/dss/dsshome1/03/ge87wod2/morphological-inflection/llm/data/gemini-2.5-flash/csb.dev"

with open (input_path,"r") as input_file, open (train_path,"w") as train_file, open (valid_path,"w") as valid_file:
	lines = input_file.readlines()
	lines.sort(key=get_word)
	distinct_words = [list(g)[0] for w, g in itertools.groupby(lines, get_word) if w != ""] # remove duplicate words

	lines.sort(key=get_lemma)
	lemmas = [(k, list(g)) for k, g in itertools.groupby(distinct_words, get_lemma) if k != ""]
	forms = {
		"train": [],
		"validation": []
	}
	validation_number = 1000
	train_number = 10000
	tables_shuffled = random.shuffle(lemmas)
	lemma_idx = 0
	for lemma, examples in lemmas:
		if(len(forms["validation"])<validation_number): 
			forms["validation"]+=examples
		else:
			forms["train"]+=examples

	forms["validation"]=forms["validation"][:validation_number]
	forms["train"]=forms["train"][:train_number]

	validation_string = "".join(forms["validation"])
	train_string = "".join(forms["train"])

	train_file.write(train_string)
	valid_file.write(validation_string)